# queries

In [4]:
import duckdb

# Composites
comp_hold  = r'holdings/*holdings.parquet'
comp_price_1h = r'composites/*1h_price.parquet'
comp_price_1d = r'composites/*1d_price.parquet'
comp_price_1wk = r'composites/*1wk_price.parquet'
comp_meta  = r'composites/*metadata.parquet'
# Constituents
consti_price_1h = r'constituents/*1h_price.parquet'
consti_price_1d = r'constituents/*1d_price.parquet'
consti_price_1wk = r'constituents/*1wk_price.parquet'
consti_meta  = r'constituents/*metadata.parquet'
consti_fin_ttm_income = r'constituents/*ttm_income_financial.parquet'
consti_fin_ttm_cashflow = r'constituents/*ttm_cashflow_financial.parquet'

def get_etf_summary():
    query = duckdb.sql(f"""
        with 
        clean_list as (
            select
                etf
                , Name as name
                , Ticker as symbol
                , SEDOL as sedol
                , Weight as weight
            from read_parquet('{comp_hold}', union_by_name=True)
            where
                sedol != '-'
        )
        select
            etf
            , count(*) as count
            , sum(weight) as weight
        from clean_list
        group by etf
        order by count desc
    """).fetchdf()
    return query

def get_etf_holdings():
    query = duckdb.sql(f"""
        with 
        clean_list as (
            select
                etf
                , Name as name
                , Ticker as symbol
                , SEDOL as sedol
                , Weight as weight
            from read_parquet('{comp_hold}', union_by_name=True)
            where
                sedol != '-'
        ),
        price as (
            select
                symbol
                , close
                , high
                , low
                , open
                , volume
                , datetime
            from read_parquet('{comp_price_1h}', union_by_name=True)
        )
        select
            * --exclude (date1, date2)
            --, case when date1 is null then date2 else date1 end as date
        from price
    """)
    return query

def get_weights():
    query = duckdb.sql(f"""
    select
        etf,
        sum(weight) * 100 as top10_weight,
        (1 - sum(weight)) * 100 as others
    from read_parquet('{compo_hold}', union_by_name=True)
    group by etf
    """).fetchdf().set_index('etf')
    return query

def get_industry_aggs():
    query = duckdb.sql(f"""
    select
        sector, 
        industry, 
        count(*) as count, 
        round(try_cast(sum(totalRevenue) as bigint), 0) as revenue, 
        round(try_cast(sum(sharesOutstanding * trailingEps) as bigint), 2) as income,
        round(try_cast(sum(sharesOutstanding * trailingEps) / sum(totalRevenue) * 100 as double), 2) as margin
    from read_parquet('{consti_meta}', union_by_name=True)
    group by sector, industry
    order by margin desc
    """).fetchdf()
    return query

def get_indicators():
    query = duckdb.sql(f"""
    with 
    master_file as (
        with
        previous_day as (
            select symbol, date,
                open, high, low, close,
                lag(open)  over (partition by symbol order by date asc) as pd_open,
                lag(high)  over (partition by symbol order by date asc) as pd_high,
                lag(low)   over (partition by symbol order by date asc) as pd_low,
                lag(close) over (partition by symbol order by date asc) as pd_close,
                volume, cast(volume * close as bigint) as value
            from read_parquet('{consti_price}', union_by_name=True)
        ),
        change as (
            select *,
                ((close / nullif(pd_close, 0)) - 1) * 100 as chg,
                ((open  / nullif(pd_close, 0)) - 1) * 100 as gap,
            from previous_day
        )
        select *,
            avg(chg)        over w AS mean,
            stddev_pop(chg) over w AS stddev
        from change
        window w as (
            partition by symbol order by date
            rows between unbounded preceding and current row
        )
    ),
    indicators as (
        select *,
            (chg - mean) / stddev as z_score
        from master_file
    )
    select *
    from indicators
    order by date desc, value desc
    """).fetchdf().set_index('symbol')
    return query

get_etf_holdings()

┌─────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────┬──────────────────────────┐
│ symbol  │       close        │        high        │        low         │        open        │ volume │         datetime         │
│ varchar │       double       │       double       │       double       │       double       │ int64  │ timestamp with time zone │
├─────────┼────────────────────┼────────────────────┼────────────────────┼────────────────────┼────────┼──────────────────────────┤
│ KBE     │  46.14500045776367 │  46.79999923706055 │             46.125 │  46.38999938964844 │ 297406 │ 2024-07-01 21:30:00+08   │
│ KBE     │ 46.064998626708984 │   46.2599983215332 │ 45.994998931884766 │  46.14500045776367 │ 151604 │ 2024-07-01 22:30:00+08   │
│ KBE     │ 46.290000915527344 │  46.29999923706055 │  46.06999969482422 │  46.06999969482422 │  75010 │ 2024-07-01 23:30:00+08   │
│ KBE     │  46.36000061035156 │  46.41630172729492 │ 46.209999084472656 │  

# model iv based signal

In [85]:
import duckdb as dk
calls = r"C:\GitHub\rlp-jym-projects\spdr-pipeline\pipeline\uploads\consti_options_calls.parquet"
puts = r"C:\GitHub\rlp-jym-projects\spdr-pipeline\pipeline\uploads\consti_options_puts.parquet"


dk.sql(f"""
with 
joined as (
    select * from read_parquet('{calls}')
    union all
    select * from read_parquet('{puts}')
),
pre as (
    select 
        * exclude (inTheMoney, contractSize, currency, contractSymbol, change)
        , impliedVolatility * (volume / sum(volume) over (partition by symbol, type)) as volWeightIV
        , impliedVolatility * (openInterest / sum(openInterest) over (partition by symbol, type)) as oiWeightIV
    from joined
),
agg as (
select
    symbol, type
    , sum(volume) as vol
    , sum(openInterest) as oi
    , round(sum(volWeightIV), 2) as volWeightIV
    , round(sum(oiWeightIV), 2) as oiWeightIV
from pre
group by symbol, type
order by symbol, type
),
flat as (
select
    symbol
    , sum(vol) as vol
    , sum(oi) as oi
    , max(case when type = 'calls' then oiWeightIV end) as callsIV
    , max(case when type = 'puts' then oiWeightIV end) as putsIV
from agg
group by symbol
),
ratio as (
select
    symbol
    , cast(oi as int) as totalOI
    , callsIV, putsIV
    , round(callsIV/putsIV, 2) as ratio
from flat
where true
    and ratio != 'nan'
    and ratio != 'inf'
order by totalOI desc
)
select *
    , case
        when ratio < 0.5 then 'high short'
        when ratio > 1 then 'unusual'
            else '' end as signal
from ratio
where true
    and totalOI > 100e3
""")
# ratio threshold and minimum oi are arbitrary
# to be validated with more data inputs
# still day 1 of options data gathering

┌─────────┬─────────┬─────────┬────────┬────────┬────────────┐
│ symbol  │ totalOI │ callsIV │ putsIV │ ratio  │   signal   │
│ varchar │  int32  │ double  │ double │ double │  varchar   │
├─────────┼─────────┼─────────┼────────┼────────┼────────────┤
│ MU      │  452540 │    0.92 │    3.3 │   0.28 │ high short │
│ MSTR    │  433872 │    1.27 │   1.48 │   0.86 │            │
│ PLTR    │  233928 │     0.7 │   0.78 │    0.9 │            │
│ INTC    │  233444 │    0.95 │    1.4 │   0.68 │            │
│ MARA    │  208998 │    0.97 │    1.0 │   0.97 │            │
│ NVDA    │  201459 │    0.44 │   0.55 │    0.8 │            │
│ SMCI    │  167914 │    1.16 │   1.24 │   0.94 │            │
│ NFLX    │  153820 │    0.56 │   0.42 │   1.33 │ unusual    │
│ MRVL    │  147015 │    1.06 │   1.65 │   0.64 │            │
│ AMD     │  139653 │    0.82 │   1.29 │   0.64 │            │
│ HOOD    │  125252 │     0.8 │   1.07 │   0.75 │            │
│ WULF    │  121412 │    1.87 │    1.2 │   1.56 │ unusu

# end

# yfinance resources

In [44]:
# HOLDINGS
    # insider_purchases
    # insider_transactions
    # insider_roster_holders
    # institutional_holders
    # mutualfund_holders

# ANALYSIS
    # revenue_estimate
    # earnings_estimate
    # growth_estimates
    # eps_revisions
    # eps_trend
    # upgrades_downgrades
    # recommendations
    # analyst_price_targets

# STOCKS
    # dividends
    # splits

# SECTORS
    # keys
        # basic-materials
        # communication-services
        # consumer-cyclical
        # consumer-defensive
        # energy
        # financial-services
        # healthcare
        # industrials
        # real-estate
        # technology
        # utilities
    # attributes
        # industries
        # name
        # overview
        # research_reports
        # symbol
        # ticker
        # top_companies
        # top_etfs
        # top_mutual_funds
        # industry-specific
           # sector_name
           # top_growth_companies
           # top_performing_companies

ticker = 'NVDA'
yf.Ticker(ticker).ttm_income_stmt.T.reset_index().to_parquet(f'{ticker}_ttm_income.parquet')
yf.Ticker(ticker).ttm_cashflow.T.reset_index().to_parquet(f'{ticker}_ttm_cashflow.parquet')
yf.Ticker(ticker).quarterly_income_stmt.T.reset_index().to_parquet(f'{ticker}_qtr_income.parquet')
yf.Ticker(ticker).quarterly_cashflow.T.reset_index().to_parquet(f'{ticker}_qtr_cashflow.parquet')
yf.Ticker(ticker).quarterly_balance_sheet.T.reset_index().to_parquet(f'{ticker}_qtr_assets.parquet')
yf.Ticker(ticker).earnings_dates.reset_index().to_parquet(f'{ticker}_dates.parquet')

In [ ]:
# ticker_list = [
#     # state street etfs
#     'XLC', 'XLY', 'XLP', 'XLE', 'XLF', 'XLV', 'XLI', 'XLB', 'XLRE', 'XAR',
#     'KBE', 'XBI', 'KCE', 'XHE', 'XHS', 'XHB', 'KIE', 'XME', 'XES', 'XOP', 
#     'XPH', 'KRE', 'XRT', 'XSD', 'XSW', 'XTL', 'XTN', 'XLK', 'XLU', 'SPY'
# ]
# # ticker_list = ['NVDA', 'TSLA', 'INTC']

# options = pd.DataFrame()
# for ticker in ticker_list:
#     optionchaincalls = yf.Ticker(ticker).option_chain().calls
#     optionchainputs = yf.Ticker(ticker).option_chain().puts
    
#     optionsx = duckdb.sql(f"""
#         with
#         agg as (
#             with 
#             merged as (
#                 select 'calls' as type, * from optionchaincalls
#                 union all
#                 select 'puts' as type, * from optionchainputs
#             ),
#             calls as (
#                 with totals as (
#                     select
#                         type,
#                         impliedVolatility as IV, 
#                         volume, openInterest as OI, 
#                         sum(volume)       over (partition by type) as totalVolume,
#                         sum(openInterest) over (partition by type) as totalInterest
#                     from merged
#                 )
#                 select *,
#                     volume / totalVolume as weightedVol,
#                     OI     / totalInterest as weightedOI 
#                 from totals
#             ),
#             weighted as (
#                 select *,
#                     IV * weightedVol as volWeightIV,
#                     IV * weightedOI as OIWeightIV
#                 from calls
#             )
#             select
#                 type,
#                 sum(volume) as volume,
#                 sum(OI) as OI,
#                 round(sum(volWeightIV), 2) as volWeightIV,
#                 round(sum(OIWeightIV), 2) as OIWeightIV
#             from weighted
#             group by type
#         ),
#         ratio as (
#             select
#                 '{ticker}' as ticker,
#                 round(
#                     max(case when type = 'calls' then volume end) / 
#                     max(case when type = 'puts'  then volume end), 2) as volume,
#                 round(
#                     max(case when type = 'calls' then OI end) / 
#                     max(case when type = 'puts'  then OI end), 2) as OI,
#                 round(
#                     max(case when type = 'calls' then volWeightIV end) / 
#                     max(case when type = 'puts'  then volWeightIV end), 2) as volWeightIV,
#                 round(
#                     max(case when type = 'calls' then OIWeightIV end) / 
#                     max(case when type = 'puts'  then OIWeightIV end), 2) as OIWeightIV
#             from agg
#         )
#         select * from ratio
#     """).fetchdf()

#     options = pd.concat([options, optionsx], ignore_index=True)   

# options

In [14]:
import duckdb
duckdb.sql("""
select *
from read_parquet('C:\GitHub\rlp-jym-projects\spdr-pipeline\pipeline\holdings\*.parquet', union_by_name=True) 
""")

<>:4: SyntaxWarning: "\G" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\G"? A raw string is also an option.
<>:4: SyntaxWarning: "\G" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\G"? A raw string is also an option.
C:\Users\ralph\AppData\Local\Temp\ipykernel_23864\2819547204.py:4: SyntaxWarning: "\G" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\G"? A raw string is also an option.
  from read_parquet('C:\GitHub\rlp-jym-projects\spdr-pipeline\pipeline\holdings\*.parquet', union_by_name=True)


IOException: IO Error: No files found that match the pattern "C:\GitHublp-jym-projects\spdr-pipeline\pipeline\holdings\*.parquet"